In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
import time
import matplotlib.pyplot as plt

from abc import ABC, abstractmethod
from typing import Dict, Any, List, Tuple

class BarrierOptimizationProblem(ABC):
    @abstractmethod
    def f_obj(self, x: np.ndarray) -> float:
        pass

    @abstractmethod
    def grad_obj(self, x: np.ndarray) -> np.ndarray:
        pass

    @abstractmethod
    def hess_obj(self, x: np.ndarray) -> np.ndarray:
        pass

    @abstractmethod
    def f_barrier(self, x: np.ndarray) -> float:
        pass

    @abstractmethod
    def grad_barrier(self, x: np.ndarray) -> np.ndarray:
        pass

    @abstractmethod
    def hess_barrier(self, x: np.ndarray) -> np.ndarray:
        pass

    @abstractmethod
    def constraints(self) -> Tuple[np.ndarray, np.ndarray]:
        pass

    @abstractmethod
    def generate_start_points(self, num_points: int) -> List[np.ndarray]:
        pass

class Optimizer(ABC):
    @abstractmethod
    def solve(self, problem: BarrierOptimizationProblem, x0: np.ndarray, **kwargs) -> Dict[str, Any]:
        """
        Должен возвращать словарь с ключами:
        - 'opt_val': оптимальное значение
        - 'opt_var': точка минимума
        - 'time': время работы
        - 'iters': количество итераций (внешних или суммарно внутренних)
        - 'status': статус решения
        - 'history': список значений функции f_obj(x) на каждой внешней итерации
        """
        pass

    def test(self, problem: BarrierOptimizationProblem, x0: np.ndarray, expected_val: float, tol: float = 1e-2, **kwargs) -> bool:
        try:
            res = self.solve(problem, x0, **kwargs)
            is_passed = abs(res['opt_val'] - expected_val) <= tol
            if not is_passed:
                print(f"[{self.__str__()}] Test Failed! Expected: {expected_val:.4f}, Got: {res['opt_val']:.4f}")
            return is_passed
        except Exception as e:
            print(f"[{self.__str__()}] Test Failed with exception: {e}")
            return False

    @abstractmethod
    def __str__(self) -> str:
        pass

class DataGenerator(ABC):
    @abstractmethod
    def generate(self, n: int) -> Tuple[np.ndarray, np.ndarray]:
        pass

    @abstractmethod
    def __str__(self) -> str:
        pass

In [ ]:
class LogOptimalPrimalWithBarrier(BarrierOptimizationProblem):
    def __init__(self, P: np.ndarray, pi: np.ndarray):
        self.P = P
        self.pi = pi
        self.n, self.m = P.shape

    def f_obj(self, x: np.ndarray) -> float:
        return -np.sum(self.pi * np.log(self.P.T @ x + 1e-16))

    def grad_obj(self, x: np.ndarray) -> np.ndarray:
        denom = self.P.T @ x + 1e-16
        return -self.P @ (self.pi / denom)

    def hess_obj(self, x: np.ndarray) -> np.ndarray:
        denom = self.P.T @ x + 1e-16
        tmp = self.pi / (denom ** 2)
        return self.P @ np.diag(tmp) @ self.P.T

    def f_barrier(self, x: np.ndarray) -> float:
        return np.sum(1.0 / (x + 1e-16))

    def grad_barrier(self, x: np.ndarray) -> np.ndarray:
        return -1.0 / (x**2 + 1e-16)

    def hess_barrier(self, x: np.ndarray) -> np.ndarray:
        return np.diag(2.0 / (x**3 + 1e-16))

    def constraints(self) -> Tuple[np.ndarray, np.ndarray]:
        A = np.ones((1, self.n))
        b = np.array([1.0])
        return A, b

    def generate_start_points(self, num_points: int) -> List[np.ndarray]:
        points = []
        points.append(np.ones(self.n) / self.n)

        for _ in range(num_points - 1):
            x = np.random.rand(self.n) + 0.1
            x /= np.sum(x)
            points.append(x)
        return points

class LogOptimalDualWithBarrier(BarrierOptimizationProblem):
    def __init__(self, P: np.ndarray, pi: np.ndarray):
        self.P = P
        self.pi = pi
        self.n, self.m = P.shape
        self.n_var = self.m + 1

    def _unpack(self, z: np.ndarray):
        return z[:self.m], z[self.m]

    def f_obj(self, z: np.ndarray) -> float:
        w, nu = self._unpack(z)
        return -np.sum(self.pi * np.log(w + 1e-16)) + nu

    def grad_obj(self, z: np.ndarray) -> np.ndarray:
        w, nu = self._unpack(z)
        grad_w = -self.pi / (w + 1e-16)
        grad_nu = 1.0
        return np.concatenate([grad_w, [grad_nu]])

    def hess_obj(self, z: np.ndarray) -> np.ndarray:
        w, nu = self._unpack(z)
        H = np.zeros((self.n_var, self.n_var))
        H[:self.m, :self.m] = np.diag(self.pi / (w**2 + 1e-16))
        return H

    def f_barrier(self, z: np.ndarray) -> float:
        w, nu = self._unpack(z)
        margin = nu - self.P @ w
        return np.sum(1.0 / (margin + 1e-16)) + np.sum(1.0 / (w + 1e-16))

    def grad_barrier(self, z: np.ndarray) -> np.ndarray:
        w, nu = self._unpack(z)
        margin = nu - self.P @ w + 1e-16

        grad_nu = -np.sum(1.0 / (margin**2))
        grad_w = self.P.T @ (1.0 / (margin**2)) - 1.0 / (w**2 + 1e-16)
        return np.concatenate([grad_w, [grad_nu]])

    def hess_barrier(self, z: np.ndarray) -> np.ndarray:
        w, nu = self._unpack(z)
        margin = nu - self.P @ w + 1e-16
        inv_margin_cube = 2.0 / (margin**3)

        H_nu_nu = np.sum(inv_margin_cube)
        H_w_nu = -self.P.T @ inv_margin_cube
        H_w_w = self.P.T @ np.diag(inv_margin_cube) @ self.P + np.diag(2.0 / (w**3 + 1e-16))

        H = np.zeros((self.n_var, self.n_var))
        H[:self.m, :self.m] = H_w_w
        H[:self.m, self.m] = H_w_nu
        H[self.m, :self.m] = H_w_nu
        H[self.m, self.m] = H_nu_nu
        return H

    def constraints(self) -> Tuple[np.ndarray, np.ndarray]:
        return np.zeros((0, self.n_var)), np.zeros(0)

    def generate_start_points(self, num_points: int) -> List[np.ndarray]:
        points = []
        for i in range(num_points):
            w = np.random.rand(self.m) * (i + 1)
            margin = np.random.rand() * 5 + 1.0
            nu = np.max(self.P @ w) + margin
            points.append(np.concatenate([w, [nu]]))
        return points

In [ ]:
class NormalDataGenerator(DataGenerator):
    def __init__(self, scale: float = 0.1, shift: float = 1.0):
        self.scale = scale
        self.shift = shift

    def generate(self, n: int) -> Tuple[np.ndarray, np.ndarray]:
        m = n * 2
        P = np.random.randn(n, m) * self.scale + self.shift
        P += np.eye(n, m) * n
        pi = np.random.rand(m)
        pi /= pi.sum()
        return P, pi

    def __str__(self) -> str:
        return f"Normal(scale={self.scale}, shift={self.shift})"

class UniformDataGenerator(DataGenerator):
    def __init__(self, low: float = 0.9, high: float = 1.1):
        self.low = low
        self.high = high

    def generate(self, n: int) -> Tuple[np.ndarray, np.ndarray]:
        m = n * 2
        P = np.random.uniform(self.low, self.high, size=(n, m))
        P += np.eye(n, m) * n
        pi = np.random.rand(m)
        pi /= pi.sum()
        return P, pi

    def __str__(self) -> str:
        return f"Uniform(low={self.low}, high={self.high})"

In [ ]:
class CVXPYOptimizer(Optimizer):
    def solve(self, problem: BarrierOptimizationProblem, x0: np.ndarray = None, **kwargs) -> Dict[str, Any]:
        # TODO: реализовать CVXPY для получения идеального ответа, чтобы удовлетворять условиям интерфейса Optimizer
        raise NotImplementedError()

    def __str__(self) -> str:
        return "CVXPY"

In [ ]:
class NewtonBarrierOptimizer(Optimizer):
    def solve(self, problem: BarrierOptimizationProblem, x0: np.ndarray, t0: float = 1.0, mu: float = 10.0, **kwargs) -> Dict[str, Any]:
        # TODO: реализовать с методом Ньютона, чтобы удовлетворял условиям интерфейса Optimizer
        raise NotImplementedError()

    def __str__(self) -> str:
        return "Barrier (Newton)"

In [ ]:
class QuasiNewtonBarrierOptimizer(Optimizer):
    def solve(self, problem: BarrierOptimizationProblem, x0: np.ndarray, t0: float = 1.0, mu: float = 10.0, **kwargs) -> Dict[str, Any]:
        # TODO: реализовать с квазиньютоновским методом, чтобы удовлетворял условиям интерфейса Optimizer
        raise NotImplementedError()

    def __str__(self) -> str:
        return "Barrier (Quasi-Newton)"

In [ ]:
class ExperimentRunner:
    def __init__(self,
                 n_list: List[int],
                 cases_per_n: int,
                 start_points_count: int,
                 ground_truth_solver: Optimizer,
                 optimizers: List[Optimizer],
                 generators: List[DataGenerator],
                 t0_list: List[float] = [1.0],
                 mu_list: List[float] = [15.0]):
        self.n_list = n_list
        self.cases_per_n = cases_per_n
        self.start_points_count = start_points_count
        self.ground_truth_solver = ground_truth_solver
        self.optimizers = optimizers
        self.generators = generators
        self.t0_list = t0_list
        self.mu_list = mu_list

    def run(self) -> pd.DataFrame:
        rows = []
        plot_data_primal = {}
        plot_data_dual = {}

        for n in self.n_list:
            for gen in self.generators:
                for case in range(self.cases_per_n):
                    P, pi = gen.generate(n)
                    primal = LogOptimalPrimalWithBarrier(P, pi)
                    dual = LogOptimalDualWithBarrier(P, pi)

                    p_start_points = primal.generate_start_points(self.start_points_count)
                    d_start_points = dual.generate_start_points(self.start_points_count)

                    gt_primal = self.ground_truth_solver.solve(primal, x0=p_start_points[0])
                    gt_dual = self.ground_truth_solver.solve(dual, x0=d_start_points[0])

                    for t0 in self.t0_list:
                        for mu in self.mu_list:
                            for sp_idx in range(self.start_points_count):
                                x0_p = p_start_points[sp_idx]
                                z0_d = d_start_points[sp_idx]
                                for opt in self.optimizers:
                                    opt.test(primal, x0_p, gt_primal['opt_val'])
                                    opt.test(dual, z0_d, gt_dual['opt_val'])

                                    try:
                                        res_primal = opt.solve(primal, x0=x0_p, t0=t0, mu=mu)
                                        res_dual = opt.solve(dual, x0=z0_d, t0=t0, mu=mu)

                                        rows.append({
                                            'n': n, 'case': case, 'generator': str(gen), 'solver': str(opt),
                                            'start_point': sp_idx, 't0': t0, 'mu': mu,
                                            'p_opt': res_primal['opt_val'], 'd_opt': res_dual['opt_val'],
                                            'time_primal': res_primal['time'], 'iters_primal': res_primal['iters'],
                                            'time_dual': res_dual['time'], 'iters_dual': res_dual['iters'],
                                        })
                                    except NotImplementedError:
                                        pass

        # TODO: добавить вывод графиков

        return pd.DataFrame(rows)

In [ ]:
if __name__ == "__main__":
    runner = ExperimentRunner(
        n_list=[10, 20],
        cases_per_n=2,
        start_points_count=2,
        ground_truth_solver=CVXPYOptimizer(),
        optimizers=[NewtonBarrierOptimizer(), QuasiNewtonBarrierOptimizer()],
        generators=[NormalDataGenerator(scale=0.1, shift=1.0)],
        t0_list=[1.0, 10.0],
        mu_list=[10.0, 20.0]
    )
    df = runner.run()
    # TODO: добавить полный вывод таблицы с данными по запускам